# Reproduce the ORIGINAL L³P paper envs (2019 stack) — one clean pass

Advisor requirement: run on the paper's own MuJoCo envs
(`LunjunZhang/world-model-as-a-graph`), not the gymnasium-robotics substitutes.
That means the 2019 stack: python 3.7.4, torch 1.5.1+cu101, tf 1.13.1, gym 0.13.1,
mpi4py, mujoco_py 2.0.2.13 + MuJoCo 2.0. All of that is built by
`repro/setup_kaggle.sh` (verified end-to-end on a Kaggle T4).

**Before you start:**
- Settings → **Accelerator = GPU T4** (torch cu101 supports T4/P100; NOT A100/H100).
- Settings → **Internet = ON**.
- ⚠️ Switching the Accelerator **restarts the session and WIPES the `l3p` env**
  (it lives in `/opt/conda/envs`, not `/kaggle/working`). Always set GPU FIRST,
  then run cells 1→2 to (re)build. `/kaggle/working` may also be wiped on restart.

**Run order:** 1 get code → 2 setup → 3 verify GPU → (4 restore, resume only) →
(5 optional smoke) → 6 train Fetch → (7 AntMaze later). Each training cell is
self-contained (re-exports PATH + LD_LIBRARY_PATH) so you can re-run any one alone.

## 1. Get code (your fork with `repro/` + the paper repo)

In [ ]:
import os
# Your fork holds repro/ (setup_kaggle.sh, kaggle_train.py). Branch: retrain.
if os.path.isdir('/kaggle/working/latent_landmarks'):
    !cd /kaggle/working/latent_landmarks && git pull -q origin retrain
else:
    !git clone -q -b retrain https://github.com/Jun1801/latent_landmarks.git /kaggle/working/latent_landmarks
# The paper repo (setup also clones this, but ensure it here too).
if not os.path.isdir('/kaggle/working/wmag'):
    !git clone -q https://github.com/LunjunZhang/world-model-as-a-graph /kaggle/working/wmag
print('code ready')

## 2. One-time setup (~10–15 min): old stack + MuJoCo 2.0
Builds the `l3p` conda env. Idempotent — re-running reuses a valid env. Re-run
this after any GPU/accelerator switch (the env gets wiped on restart).

In [ ]:
!bash /kaggle/working/latent_landmarks/repro/setup_kaggle.sh

## 3. Verify GPU + env
`cuda True` means the accelerator is active AND the env survived. If the GPU line
says NO GPU → enable Accelerator = GPU T4 (that restarts the session; then re-run
cells 1–2). If `import` fails → the env was wiped, re-run cell 2.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo ">>> NO GPU: enable Accelerator = GPU T4 in Settings (restarts session -> re-run cells 1-2)"
!export PATH=/opt/conda/bin:$PATH; \
 export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-}; \
 conda run -n l3p python -c "import torch, mujoco_py; print('cuda', torch.cuda.is_available(), '| mujoco', mujoco_py.__version__)" 

## 4. (Resume only) restore checkpoints from a previous session
`/kaggle/working` is wiped between sessions, so to CONTINUE training you must have
committed the previous session's output and attached it here as a Dataset.
`SRC` is a placeholder — it is the Dataset YOU create from a prior session's
Output (see the last cell). On the very first run there is nothing to restore and
this prints "starting fresh" — that is correct, just move on.

In [ ]:
import os, shutil
SRC = '/kaggle/input/l3p-experiments/experiments'   # <-- YOUR dataset from a previous session's Output
DST = '/kaggle/working/experiments'
if os.path.isdir(SRC):
    shutil.copytree(SRC, DST, dirs_exist_ok=True); print('restored ->', DST)
else:
    print('no prior experiments/ attached — starting fresh (correct on the first session)')

## 5. (OPTIONAL) quick pipeline smoke — PointMaze, ~2 min
Sanity-check that training + checkpoint-save work before a long run. PointMaze is
the paper's easy env; a short run already reaches ~0.95+ plan success. Skip this
cell if you just want to train Fetch.

In [ ]:
%%bash
export PATH=/opt/conda/bin:$PATH
export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-}
cd /kaggle/working/wmag
conda run -n l3p python /kaggle/working/latent_landmarks/repro/kaggle_train.py \
  --env-name PointMaze-v1 --test-env-name PointMazeTest-v1 \
  --ckpt-name smoke_pm --seed 123 --n-epochs 20 \
  --save-dir /kaggle/working/experiments

## 6. Train Fetch (FetchPickAndPlace) — resume-aware
Fetch uses a DIFFERENT entry point (`rl.main_latent_fetch`) and the paper's exact
flags (from `scripts/pick.sh`). The default `--n_epochs` is 10000 (~330h) — an
upper bound, NOT a target; Fetch pick converges far sooner, so it is capped to 200
here. Watch `Test_TestEnv_PlanSuccessRate` climb; **stop when it plateaus**.

Resume: checkpoints save every epoch to `.../fetch_s967/state/`. Re-running this
cell auto-adds `--resume_ckpt` (restores weights/optimizer/replay/total_timesteps;
the epoch counter restarts at 0 but training continues). The paper uses 3 seeds
(967/837/645) — do one at a time (change `CKPT`/`--seed`).

⚠️ `replay_0.pt` is ~700 MB. It MUST be persisted for resume — see the last cell.

In [ ]:
%%bash
export PATH=/opt/conda/bin:$PATH
export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-}
cd /kaggle/working/wmag
SAVE=/kaggle/working/experiments; CKPT=fetch_s967
RESUME=""
if [ -f "$SAVE/FetchPickAndPlace-v1/$CKPT/state/algo.pt" ]; then
  RESUME="--resume_ckpt $CKPT"; echo "[resume] continuing $CKPT"
else
  echo "[fresh] new run $CKPT"
fi
conda run -n l3p python -m rl.main_latent_fetch \
  --env_name FetchPickAndPlace-v1 --test_env_name FetchPickAndPlace-v1 --cuda \
  --seed 967 --n_cycles 10 --clip_inputs --normalize_inputs --gamma 0.99 \
  --n_initial_rollouts 100 --n_test_rollouts 10 --plan_eps 0.5 \
  --n_latent_landmarks 80 --latent_batch_size 150 --n_extra_landmark 20 \
  --dist_clip -15.0 --start_planning_n_traj 6000 --use_forward_empty_step \
  --n_epochs 200 --save_dir $SAVE --ckpt_name $CKPT $RESUME

## 7. (Later) AntMaze — the heaviest env
AntMaze uses `rl.main_latent` (so `kaggle_train.py` fits it). It is far heavier
than Fetch — locomotion bootstraps slowly and the paper runs 4 seeds. Expect many
resumed sessions (or rent a cloud GPU with `repro/Dockerfile`, no 12h cap).
Verify the exact flags against the paper's `scripts/train_antmaze.sh` before a long
run. Do Fetch first; only come here once Fetch is a confirmed result.

In [ ]:
%%bash
export PATH=/opt/conda/bin:$PATH
export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-}
cd /kaggle/working/wmag
# Paper-faithful AntMaze config (scripts/train_antmaze.sh, seed 221), resume-aware.
# n_epochs 20000 is a nominal cap -- resume restarts the epoch loop at 0 but keeps
# the restored weights/replay, so just re-run each session and kill when eval plateaus.
SAVE=/kaggle/working/experiments; CKPT=antmaze_s221_paper
RESUME=""
if [ -f "$SAVE/AntMaze-v1/$CKPT/state/algo.pt" ]; then
  RESUME="--resume_ckpt $CKPT"; echo "[resume] continuing $CKPT"
else
  echo "[fresh] new run $CKPT"
fi
conda run -n l3p python -m rl.main_latent \
  --n_workers 3 --env_name AntMaze-v1 --test_env_name AntMazeTest-v1 \
  --gamma 0.98 --n_epochs 20000 --clip_return 100 --future_step 100 \
  --n_extra_landmark 150 --dist_clip -20.0 --n_latent_landmarks 50 \
  --latent_batch_size 256 --batch_size 1000 --cuda \
  --grad_value_clipping -1.0 --grad_norm_clipping 15.0 --action_l2 0.05 \
  --optimize_every 2 --seed 221 \
  --save_dir $SAVE --ckpt_name $CKPT $RESUME

## 8. Persist before the session ends (so you can resume)
`/kaggle/working` is saved as the notebook **Output** only when you **Save Version
/ Commit**. To continue next session:
1. Stop the training cell, then **Save Version** (commits `/kaggle/working`, incl.
   the ~700 MB replay).
2. Next session: **Add Data → Your Work / Notebook Output** → attach this output.
3. Set `SRC` in cell 4 to `/kaggle/input/<that-dataset>/experiments`, run cell 4,
   then re-run the training cell → it prints `[resume]`.

**Verify a checkpoint exists before committing:**

In [ ]:
!ls -la /kaggle/working/experiments/FetchPickAndPlace-v1/fetch_s967/state/ 2>/dev/null || echo 'no Fetch checkpoint yet'

### Operating notes
- **Kaggle limits:** ~12h/session, ~30h/week GPU. Fetch (one seed, capped 200
  epochs) can often finish in one session; AntMaze needs many.
- **Resume caveat:** weights/optimizer/replay/total_timesteps ARE restored; the
  epoch loop restarts at 0 → it keeps learning. **Kill when eval plateaus.**
- **Don't trust silence:** if `Test_TestEnv_PlanSuccessRate` stays ~0 for many
  epochs after planning turns on (`start_planning_n_traj` reached), stop & inspect.
- **cuda False?** you are on CPU — training still runs but is much slower; enable
  the GPU accelerator (and re-run setup after the restart).